In [15]:
import ee
import geemap
import pandas as pd
import numpy as np
import json

import time
import io

import rasterio
import requests


In [16]:
cloud_project = "hedgementation"

try:
    ee.Initialize(project=cloud_project)
except:
    ee.Authenticate()
    ee.Initialize(project=cloud_project)

In [17]:
import datetime


nb_pixel = 128
scale = 10
selected_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']

start_year = "2017"
end_year = "2018"

datetime_format = "%Y%m%d"

start_date = datetime.datetime.strptime(f"{start_year}0917", datetime_format)
end_date = datetime.datetime.strptime(f"{end_year}1027", datetime_format)

date_range = [start_date + datetime.timedelta(days=i * 5) 
              for i in range(int((end_date - start_date).days / 5))]

with open("../metadata.geojson", "rb+") as f:
    metadata = json.load(f)
tiles = list(set([feature["properties"]["TILE"] for feature in metadata["features"]]))

destination_folder = "/content/gdrive/My Drive/ExportTest"

s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filter(ee.Filter.date(
    ee.Date(start_date.strftime("%Y-%m-%d")),
    ee.Date(end_date.strftime("%Y-%m-%d"))
)).select(selected_bands)

In [18]:
def find_nearest_pixel(coords, 
                       x_tile, 
                       y_tile, 
                       initial_crs='EPSG:2154', 
                       test=True):
    
    initial_point = ee.Geometry.Point(ee.List(coords), proj=initial_crs)
    crs_4326_coords = initial_point.transform(ee.Projection('EPSG:4326'), 0).coordinates()
    
    lon = ee.Number(crs_4326_coords.get(0))
    lat = ee.Number(crs_4326_coords.get(1))
    
    zoneNumber = lon.add(180).divide(6).floor().add(1)
    epsgNorth = ee.Number(32600).add(zoneNumber)
    epsgSouth = ee.Number(32700).add(zoneNumber)
    epsgCode = ee.Algorithms.If(lat.gte(0), epsgNorth, epsgSouth)
    proj = ee.String("EPSG:").cat(ee.String(epsgCode))
    
    utm_coords = ee.Geometry.Point(ee.List(crs_4326_coords)).transform(
        ee.Projection(proj), 0
    ).coordinates()
    
    point_x = ee.Number(utm_coords.get(0))
    point_y = ee.Number(utm_coords.get(1))
    
    if test:
        return (point_x, point_y), proj.getInfo()
    
    delta_x = point_x.subtract(x_tile)
    delta_y = point_y.subtract(y_tile)
    
    nearest_point_x = delta_x.divide(scale).round().multiply(scale).add(x_tile)
    nearest_point_y = delta_y.divide(scale).round().multiply(scale).add(y_tile)
    
    return nearest_point_x, nearest_point_y

def compute_corner_utm_from_centroide(coords, proj):
    pixel_size = ee.Number(scale)
    patch_size = ee.Number(nb_pixel)
    half_size = pixel_size.multiply(patch_size).divide(2)

    x = ee.Number(coords[0])
    y = ee.Number(coords[1])

    x_min = x.subtract(half_size)
    y_min = y.subtract(half_size)
    x_max = x.add(half_size)
    y_max = y.add(half_size)

    
    square_polygon = ee.Geometry.Polygon([[
        [x_min, y_min],  
        [x_max, y_min],  
        [x_max, y_max],  
        [x_min, y_max],  
        [x_min, y_min]   
    ]], proj=ee.Projection(proj), geodesic=False)

    return square_polygon

def load_patch_coords_from_metadata(metadata, tile, test=False):
    metadata_coords = []
    ids = []

    image_tile = s2.filter(ee.Filter.eq("MGRS_TILE", tile[1:].upper())).first()
    proj = image_tile.select('B2').projection()
    x, y = proj.getInfo()['transform'][2], proj.getInfo()['transform'][5] 


    for feature in metadata["features"]:
        if feature["properties"]["TILE"] == tile:
            geometry = feature["geometry"]
            if geometry["type"] == "MultiPolygon":
                for polygon in geometry["coordinates"]:
                    for ring in polygon:
                      metadata_coords.append(ring)
                      ids.append(feature["properties"]["id"])
            elif geometry["type"] == "Polygon":
                for ring in geometry["coordinates"]:
                    metadata_coords.append(ring)
                    ids.append(feature["properties"]["id"])
        if test and len(metadata_coords) == 10:
            break

    bounds_list = []
    for coords in metadata_coords:
        mean_coords = np.array(coords)[:,:4].mean(axis=0).tolist()

        nearest_pixel_coords, proj = find_nearest_pixel(mean_coords,x,y)
        bounds = compute_corner_utm_from_centroide(nearest_pixel_coords, proj)
        bounds_list.append(bounds)
    return [
        {"id": id, 
         "bounds": bounds,
        }
        for id,bounds in zip(ids, bounds_list)
    ]

patches = load_patch_coords_from_metadata(metadata, tiles[1], test=True)

In [19]:
def get_bounding_box(patch, crs=""):
    patch_bounds = patch["bounds"]
    return patch_bounds

def stack_images(img_collection):
    img_collection = img_collection.sort('system:time_start')
    images_list = img_collection.toList(img_collection.size())
    first = ee.Image(images_list.get(0))
    rest = ee.List(images_list.slice(1))

    def stack(img, previous):
        return ee.Image(previous).addBands(ee.Image(img))

    return ee.Image(rest.iterate(stack, first))

def rename_bands(img):
    date = img.date().format('YYYY-MM-DD')
    band_names = img.bandNames().map(lambda x: ee.String(date).cat('_').cat(x))
    img = img.rename(band_names)
    return img

def get_stacked_img_for_patch(patch,image_collection, crs):
    bounding_box = get_bounding_box(patch)

    filtered_collection = image_collection.filterBounds(
        bounding_box
        ).filter(
        ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)
        )

    stacked_image = stack_images(filtered_collection.map(rename_bands))
    return stacked_image





In [20]:

crs = "EPSG:4326"

test_patch = patches[0]

bounding_box = test_patch["bounds"]
bounding_box.getInfo()

EEException: Projection: The CRS of a map projection could not be parsed.

In [ ]:

Map = geemap.Map()

Map.addLayer(bounding_box, {"color":"FF0000"}, name=f"target poly ")
Map.centerObject(bounding_box,zoom=15)

Map

EEException: Projection: The CRS of a map projection could not be parsed.

In [ ]:
img = get_stacked_img_for_patch(test_patch, s2, crs)
img

In [ ]:
url = img.getDownloadURL({
        'scale': scale,
        'crs': crs,
        'format': 'GEO_TIFF',
        'region': get_bounding_box(test_patch, crs=crs)
    })

response = requests.get(url)

tiff_bytes = io.BytesIO(response.content)
with rasterio.open(tiff_bytes) as infile:
    img_array = np.array(infile.read())
    img_array

In [ ]:
img_array.shape

(710, 129, 177)

In [ ]:
def export_patch(patch, img, crs, scale=10):
    patch_bounds = patch["bounds"]

    id = patch_bounds["id"]
    bounding_box = ee.Geometry.Rectangle(
        [[patch_bounds['x_min'], patch_bounds['y_min']],
        [patch_bounds['x_max'], patch_bounds['y_max']]],
        geodesic=False,
        proj=crs
    )

    task = ee.batch.Export.image.toDrive(
            image=img.clip(bounding_box),
            description=f'export_{id}',
            folder=destination_folder,
            fileNamePrefix=f'stacked_{id}',
            region=bounding_box,
            scale=scale,
            maxPixels=1e10,
            crs=crs,
            fileFormat='GeoTIFF'
        )

    task.start()
    return task